# Gemma 2 9B IT — Full Watermarking Pipeline

Runs the complete experiment pipeline for `google/gemma-2-9b-it`.

**Prerequisites:**
- A100 or T4 GPU runtime (Runtime → Change runtime type → GPU)
- HuggingFace account with Gemma 2 license accepted at huggingface.co/google/gemma-2-9b-it
- HuggingFace access token

**Can run in parallel with `colab_llama.ipynb` in a separate Colab session.**

**Runtime estimate:** ~3-5 hours total on T4, ~2-3 hours on A100.

## Setup

Paste your HuggingFace token below, then run this cell.

In [1]:
import os, sys

HF_TOKEN   = ''   # paste your hf_... token here (do not save/commit with token filled in)
GITHUB_URL = 'https://github.com/AliHasan-786/llm-invisible-watermarking.git'
BRANCH     = 'AmmarChanges'
REPO_DIR   = '/content/llm-invisible-watermarking'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {BRANCH}...')
    os.system(f'git clone --branch {BRANCH} {GITHUB_URL} {REPO_DIR}')
else:
    print('Pulling latest...')
    os.system(f'cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

print('Installing packages...')
os.system('pip install -q transformers datasets accelerate scipy numpy matplotlib tqdm sentencepiece protobuf huggingface_hub')
print('✓ Packages installed')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ Logged into HuggingFace')
else:
    print('⚠️  HF_TOKEN is empty — paste your token above and re-run this cell')

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {name}  ({vram:.1f} GB VRAM)')
    if vram < 18:
        print('  ⚠️  <18 GB VRAM — Gemma 2 9B needs ~18 GB fp16. A100 recommended.')
else:
    print('✗ No GPU — switch to a GPU runtime')

Cloning AmmarChanges...
Working dir: /content/llm-invisible-watermarking
Installing packages...
✓ Packages installed


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Logged into HuggingFace
✓ GPU: NVIDIA A100-SXM4-40GB  (42.4 GB VRAM)


In [2]:
import shutil, glob

# Mount Google Drive — results saved here survive disconnects.
# Re-run this cell on reconnect to restore previous progress automatically.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS = '/content/drive/MyDrive/watermark_gemma_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs('results', exist_ok=True)

def checkpoint():
    saved = []
    for fp in glob.glob('results/*gemma*') + glob.glob('results/corpus_gemma*'):
        shutil.copy2(fp, DRIVE_RESULTS)
        saved.append(os.path.basename(fp))
    if saved:
        print(f'✓ Checkpointed {len(saved)} file(s) to Drive')

# Restore any files saved from a previous session
restored = []
for fp in glob.glob(f'{DRIVE_RESULTS}/*'):
    dst = os.path.join('results', os.path.basename(fp))
    if not os.path.exists(dst):
        shutil.copy2(fp, dst)
        restored.append(os.path.basename(fp))

if restored:
    print(f'✓ Restored {len(restored)} file(s) from Drive: {restored}')
else:
    print('✓ Drive mounted — no previous results to restore')
print(f'Checkpoint folder: {DRIVE_RESULTS}')

ValueError: mount failed

---
## Step 1 — Generate corpus
**~1-2 hours. Fully resumable.**

67 prompts × 3 datasets × 2 (watermarked + control) = ~400 completions.  
Output: `results/corpus_gemma_d2.jsonl`

(Fewer prompts than LLaMA since Gemma is the secondary model — enough for meaningful comparison.)

In [ ]:
!python scripts/generate_corpus.py --model google/gemma-2-9b-it --n-per-dataset 67

In [ ]:
from pipeline.generate import load_corpus
import numpy as np
corpus = load_corpus('results/corpus_gemma_d2.jsonl')
wm  = [x for x in corpus if x['watermarked']]
uwm = [x for x in corpus if not x['watermarked']]
print(f'Watermarked:   {len(wm)}')
print(f'Unwatermarked: {len(uwm)}')
print(f'Avg tokens — wm: {np.mean([x["n_tokens"] for x in wm]):.0f},  uwm: {np.mean([x["n_tokens"] for x in uwm]):.0f}')
print(f'Sources: {set(x["source"] for x in corpus)}')
assert len(wm) > 100
print('\n✅  Corpus looks good')

In [ ]:
checkpoint()  # saves corpus_gemma_d2.jsonl to Drive

---
## Step 2 — Detection + perplexity eval
**~20 min.**

Output: `results/detection_gemma_summary.json` + `results/detection_gemma_zscores.npz`

In [ ]:
!python scripts/eval_detection.py --model google/gemma-2-9b-it

In [ ]:
import json
with open('results/detection_gemma_summary.json') as f:
    s = json.load(f)
print(f"Calibrated threshold:  {s['calibrated_z_threshold']:.3f}")
print(f"TPR (all lengths):     {s['tpr_at_1pct_fpr_all']:.1%}")
long_key = next((k for k in s if 'ge150' in k), None)
if long_key:
    v = s[long_key]
    status = '✅' if v > 0.90 else '❌'
    print(f"TPR (>=150 tokens):    {v:.1%}  {status}  (need >0.90)")
print(f"PPL ratio (wm/uwm):    {s['ppl_ratio_wm_over_uwm']:.3f}  (ideally close to 1.0)")

In [ ]:
checkpoint()  # saves detection_gemma_summary.json + zscores.npz to Drive

---
## Step 3 — Length curves
**~10 min. No model needed.**

Output: `results/length_curves_gemma.json`

In [ ]:
!python scripts/eval_length.py --model google/gemma-2-9b-it

In [ ]:
import json
with open('results/length_curves_gemma.json') as f:
    lc = json.load(f)
print(f"{'tokens':>8}  {'TPR':>6}")
for row in lc:
    bar = '█' * int(row['tpr'] * 25)
    print(f"{row['n_tokens']:>8}  {row['tpr']:>6.3f}  {bar}")

In [ ]:
checkpoint()  # saves length_curves_gemma.json to Drive

---
## Step 4 — Robustness sweep
**~15 min basic / ~90 min with LLM paraphrase.**

Output: `results/robustness_gemma.json`

In [ ]:
# Basic: word substitution + token insert/delete (~15 min)
!python scripts/eval_robustness.py --model google/gemma-2-9b-it

# Uncomment to also run LLM paraphrase attack (~90 min):
# !python scripts/eval_robustness.py --model google/gemma-2-9b-it --with-paraphrase

In [ ]:
import json
with open('results/robustness_gemma.json') as f:
    rob = json.load(f)
print(f"{'Condition':<32} {'TPR':>6}")
for cond, vals in rob.items():
    bar = '█' * int(vals['tpr_at_1pct_fpr'] * 25)
    print(f"{cond:<32} {vals['tpr_at_1pct_fpr']:>6.1%}  {bar}")

In [ ]:
checkpoint()  # saves robustness_gemma.json to Drive

---
## Step 5 — Delta sweep
**~1-2 hours.**

Output: `results/delta_sweep_gemma.json`

In [ ]:
!python scripts/eval_delta_sweep.py --model google/gemma-2-9b-it

In [ ]:
import json
with open('results/delta_sweep_gemma.json') as f:
    sweep = json.load(f)
print(f"{'delta':>7} {'TPR':>7} {'PPL_wm':>8} {'ratio':>7}")
for row in sweep:
    print(f"{row['delta']:>7.1f} {row['tpr']:>7.3f} {row['mean_ppl_wm']:>8.1f} {row['ppl_ratio']:>7.3f}")

In [ ]:
checkpoint()  # saves delta_sweep_gemma.json to Drive

---
## Download results

**Run before closing the session** — Colab deletes `/content/` when the session ends.

Downloads a zip of `results/` to your local machine. Upload this zip to the `colab_combined` notebook when both models are done.

In [ ]:
import zipfile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
archive   = f'/content/results_gemma_{timestamp}.zip'

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in Path('results').rglob('*'):
        if fp.is_file():
            zf.write(fp)

size_mb = Path(archive).stat().st_size / 1e6
print(f'Created {archive}  ({size_mb:.1f} MB)')

from google.colab import files
files.download(archive)